# Experiment 7: all-fish transition/static raster

This notebook follows the shared all-fish workflow used for experiments 1 and 5. It loads the experiment 7 cohort, validates its stimulus map, and generates z-score rasters, mean traces, left/right activity diagnostics, motion-period counts, and lifetime-sparseness figures.

Fish IDs, data location, blocks, and stimulus order come from `archive/notebooks/calcium/Exp7_trans_stat_several.ipynb`. The 25-second post-onset window follows its later configuration cell (the earlier cell used 32 seconds). The 8-second motion onset agrees with the experiment 7 stimulus configuration; the 6-second response extension follows the shared workflow.

Edit `USER_SETTINGS` and the figure controls beside each plot. With `data_base=None`, the shared loader discovers this computer's OneDrive Lab folders automatically. For a manual override, set `data_base` to the parent of the university OneDrive folder, such as `Path.home()` here or `Path(r'D:\Alejandro\Data')` on the other computer. All five fish must have the processed traces, significant-raster caches, and kept-neuron indices required by the shared loader. Reports are disabled until a run has been reviewed.


## 1. Setup and imports

In [ ]:
# Cell 01 - Setup
from pathlib import Path
import sys
import importlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

repo_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'src').is_dir())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import src.plotting_common as plot_common
import src.several_fish_figures as several_figures
import src.several_fish_loading as several_loading
import src.several_fish_reporting as several_reporting
for module in (plot_common, several_figures, several_loading, several_reporting):
    importlib.reload(module)


## 2. Experiment 7 settings

In [ ]:
# Experiment 7 cohort and plotting settings from the archived several-fish notebook.
USER_SETTINGS = {
    'experiment_name': 'Exp_7_trans_stat',
    'analysis_label': 'Exp 7 transition/static',
    'data_base': None,  # Auto-discover OneDrive; use a parent Path for a manual override.
    'fish_ids': ['L742_f03', 'L742_f05', 'L742_f06', 'L742_f09', 'L742_f10'],
    'selected_blocks': ['B1', 'B2', 'B3', 'B4'],
    'verbose_loading': False,
    'timing': {
        'fps_2p': 2.0, 't_pre_s': 5.0, 't_post_s': 25.0,
        'motion_onset_s': 8.0, 'tau_s': 6.0,
    },
    # Preserve the archive's right-side group followed by its left-side group.
    'stim_order': [14, 9, 11, 13, 4, 6, 2, 7, 8, 10, 12, 3, 5, 1],
    'combine_mode': 'mean',  # 'mean' or 'concat'.
    'trace_type': 'zscore',
    'raster_sort_mode': 'left_right_index',  # 'corravg' matches the archived raster.
    'left_control': 7,    # LeBcontrol
    'right_control': 14,  # RiBcontrol
}

# Check the archive's ID/name contract against the loaded experiment, not another experiment.
EXPECTED_STIMULUS_MAP = {
    'DF300Le': 1, 'DF300Ri': 2, 'F300Le': 3, 'F300Ri': 4,
    'F600Le': 5, 'F600Ri': 6, 'LeBcontrol': 7,
    'R300Le': 8, 'R300Ri': 9, 'R600Le': 10, 'R600Ri': 11,
    'RF600Le': 12, 'RF600Ri': 13, 'RiBcontrol': 14,
}

REPORT_SETTINGS = {
    'save_report': False,
    'run_label': 'exp7_trans_stat_all_fish_raster',
    'report_format': 'html',
    'export_notebook': True,
    'fallback_to_html': True,
    'comments': 'Enable only after reviewing the Exp 7 figures.',
}


## 3. Load and validate fish data

In [ ]:
# Load every configured fish and validate the experiment 7 stimulus mapping.
raster_inputs = several_loading.load_and_preflight_fish_raster_inputs(USER_SETTINGS)
globals().update(raster_inputs)
for fish_id, fish_data in all_fish_data.items():
    if any(fish_data['stimuli_id_map'].get(name) != stimulus_id
           for name, stimulus_id in EXPECTED_STIMULUS_MAP.items()):
        raise ValueError(f"Unexpected Exp 7 stimulus map for {fish_id}: {fish_data['stimuli_id_map']}")
stimuli_colors, stimuli_linestyles = plot_common.build_stimulus_style_maps(stimuli_names=stimuli_names)
print('Data root:', main_path)
print('Analysis root:', analysis_path)
print('Detected Exp 7 stimulus map:', stimuli_id_map)


## 4. All-fish z-score raster

In [ ]:
# All-fish z-score raster in the configured stimulus order.
FIGSIZE = (13, 8)
raster_result = several_figures.build_all_fish_raster_figure(
    all_fish_data, fish_ids, stim_order, reference_fish, timing, settings,
    stimuli_colors, stimuli_linestyles, figsize=FIGSIZE,
)
flat_matrix_all_fish = raster_result['flat_matrix_all_fish']
plt.show()


## 5. Mean z-score traces

In [ ]:
# Mean z-score traces in the same experiment 7 stimulus order.
FIGSIZE = (12, 4)
mean_trace_result = several_figures.build_plot_all_fish_mean_zscore_traces(
    all_fish_data, fish_ids, stim_order, reference_fish, timing,
    stimuli_colors, stimuli_linestyles, figsize=FIGSIZE,
)
plt.show()


## 6. Left/right activity diagnostics

In [ ]:
# Matched left/right groups from the archive's stimulus map.
OVERLAP_FIGSIZE = (5, 5)
RASTER_FIGSIZE = (16, 8)
OVERLAP_STIMULI = {
    'left': [7, 8, 10, 12, 3, 5, 1],
    'right': [14, 9, 11, 13, 4, 6, 2],
}
overlap_results = several_figures.plot_left_right_active_overlap_diagnostics(
    all_fish_data, fish_ids, reference_fish, timing, OVERLAP_STIMULI,
    motion_onset_s=timing['motion_onset_s'], tau_s=timing['tau_s'],
    show_overlap=True, show_significant_raster=True,
    overlap_figsize=OVERLAP_FIGSIZE, raster_figsize=RASTER_FIGSIZE,
)


## 7. Motion-period active-neuron counts

In [ ]:
# Motion-period active-neuron counts, preserving the configured stimulus order.
FIGSIZE = (8, 3.5)
motion_active_result = several_figures.plot_motion_active_neuron_counts(
    all_fish_data, fish_ids, reference_fish, timing, stim_order, stimuli_colors,
    figsize=FIGSIZE, motion_onset_s=timing['motion_onset_s'], tau_s=timing['tau_s'],
)
active_neuron_counts = motion_active_result['counts']


## 8. Lifetime sparseness and high-sparseness raster

In [ ]:
# Lifetime sparseness across the selected experiment 7 stimuli.
SPARSENESS_FIGSIZE = (7, 4.5)
RASTER_FIGSIZE = (13, 8)
SPARSENESS_STIMULI = stim_order  # Edit to select a subset of IDs or names.
HIGH_LIFETIME_SPARSENESS = 0.75
HIGH_SPARSENESS_COMBINE_MODE = 'mean'
sparseness_result = several_figures.plot_lifetime_sparseness_analysis(
    all_fish_data, fish_ids, reference_fish, timing, SPARSENESS_STIMULI,
    HIGH_LIFETIME_SPARSENESS, stimuli_colors, stimuli_linestyles,
    combine_mode=HIGH_SPARSENESS_COMBINE_MODE,
    sparseness_figsize=SPARSENESS_FIGSIZE, raster_figsize=RASTER_FIGSIZE,
    analysis_label=settings['analysis_label'],
)


## 9. Optional analysis report

In [ ]:
# Cell 09 - Optional reviewed-run report
# Save this notebook before enabling REPORT_SETTINGS['save_report'] so the HTML includes current outputs.
notebook_path = repo_root / 'notebooks' / 'calcium' / 'exp_07_trans_stat' / '01_all_fish_raster.ipynb'
report_summary = pd.DataFrame([
    {
        'fish_id': fish_id,
        'kept_neurons': len(fish_data['kept_neuron_indices']),
        'detected_stimuli': len(fish_data['stimuli_ids']),
    }
    for fish_id, fish_data in all_fish_data.items()
])
report_result = several_reporting.save_analysis_report_run(
    settings=settings,
    report_settings=REPORT_SETTINGS,
    analysis_path=analysis_path,
    tables={'run_summary': report_summary},
    notebook_path=notebook_path,
    report_name='01_all_fish_raster_report',
    extra_metadata={
        'stim_order': stim_order,
        'combine_mode': settings['combine_mode'],
        'trace_type': settings['trace_type'],
        'raster_sort_mode': settings['raster_sort_mode'],
        'all_fish_matrix_shape': flat_matrix_all_fish.shape,
    },
)
report_result
